# 12. The Berth Allocation with Tidal Windows Problem
## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Goal
Formulate and solve the berth allocation problem with tidal constraints using mathematical optimization techniques.

### Key Assumptions
- Tidal windows are predictable and follow regular patterns
- Vessel draft requirements must be satisfied at service time
- Berths have different depth characteristics based on tidal conditions
- Processing times are deterministic and known in advance

### Approach (Step-by-Step)
1. **Problem Modeling**: Define sets, parameters, and decision variables
2. **Mathematical Formulation**: Create objective function and constraints
3. **Convex Relaxation**: Transform binary variables to continuous for tractability
4. **Solution Method**: Use interior-point methods for convex optimization
5. **Result Analysis**: Evaluate solution quality and feasibility

### What to Look for in the Results
- Optimal berth assignments respecting tidal constraints
- Service start times within feasible tidal windows
- Minimal total weighted completion time
- Satisfaction of all draft and non-overlap constraints

### Concrete Example (from the source)
Port of Hamburg scenario with 3 vessels over 24-hour period:
- Vessel A: Draft 12m, Processing time 4h, Arrival 06:00
- Vessel B: Draft 14m, Processing time 6h, Arrival 08:00
- Vessel C: Draft 11m, Processing time 3h, Arrival 14:00
- High tides at 07:30 and 20:00 (3-hour windows each)

In [ ]:
# Import required packages for mathematical optimization
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.optimize import minimize, LinearConstraint, Bounds
from dataclasses import dataclass
from typing import List, Tuple
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

In [ ]:
@dataclass
class Vessel:
    """Represents a vessel with its characteristics"""
    id: int
    draft: float  # Required draft depth in meters
    processing_time: float  # Loading/unloading duration in hours
    arrival_time: float  # Earliest arrival time in hours from start
    demurrage_cost: float  # Cost per hour of delay
    
@dataclass
class Berth:
    """Represents a berth position with its characteristics"""
    id: int
    length: float  # Length of berth in meters
    depth_low_tide: float  # Available depth at low tide
    depth_high_tide: float  # Available depth at high tide
    
@dataclass
class TidalWindow:
    """Represents a tidal window with time constraints"""
    start_time: float  # Window start time in hours
    end_time: float  # Window end time in hours
    depth_factor: float  # Tidal depth factor (0=low tide, 1=high tide)

In [ ]:
# Define the concrete example from the problem statement
vessels = [
    Vessel(id=0, draft=12.0, processing_time=4.0, arrival_time=6.0, demurrage_cost=8000),
    Vessel(id=1, draft=14.0, processing_time=6.0, arrival_time=8.0, demurrage_cost=12000),
    Vessel(id=2, draft=11.0, processing_time=3.0, arrival_time=14.0, demurrage_cost=6000)
]

berths = [
    Berth(id=0, length=300.0, depth_low_tide=10.0, depth_high_tide=15.0),
    Berth(id=1, length=350.0, depth_low_tide=11.0, depth_high_tide=16.0)
]

# Define tidal windows for 24-hour period
tidal_windows = [
    TidalWindow(start_time=6.5, end_time=9.5, depth_factor=1.0),  # High tide at 07:30
    TidalWindow(start_time=19.0, end_time=22.0, depth_factor=1.0)  # High tide at 20:30
]

print(f"Problem setup: {len(vessels)} vessels, {len(berths)} berths, {len(tidal_windows)} tidal windows")
print(f"Vessels: {[(v.id, v.draft, v.processing_time, v.arrival_time) for v in vessels]}")
print(f"Berths: {[(b.id, b.depth_low_tide, b.depth_high_tide) for b in berths]}")

In [ ]:
def get_tidal_depth(time: float, berth: Berth) -> float:
    """Calculate available depth at a given time for a specific berth"""
    # Check if time is within any tidal window
    for window in tidal_windows:
        if window.start_time <= time <= window.end_time:
            # High tide period
            return berth.depth_high_tide
    
    # Low tide period
    return berth.depth_low_tide

def is_feasible_assignment(vessel: Vessel, berth: Berth, start_time: float) -> bool:
    """Check if vessel assignment is feasible"""
    # Check arrival time constraint
    if start_time < vessel.arrival_time:
        return False
    
    # Check draft constraint at service time
    available_depth = get_tidal_depth(start_time, berth)
    if available_depth < vessel.draft:
        return False
    
    # Check if entire service period is within tidal window
    end_time = start_time + vessel.processing_time
    in_tidal_window = False
    for window in tidal_windows:
        if window.start_time <= start_time and end_time <= window.end_time:
            in_tidal_window = True
            break
    
    return in_tidal_window

# Test feasibility function
print("Testing feasibility checks:")
for v in vessels:
    for b in berths:
        feasible_times = []
        for t in np.arange(v.arrival_time, 24, 0.5):
            if is_feasible_assignment(v, b, t):
                feasible_times.append(t)
        print(f"Vessel {v.id} at Berth {b.id}: feasible start times {feasible_times[:3]}...")

In [ ]:
def objective_function(x):
    """Objective function for convex optimization
    
    x vector format: [x_0,0, x_0,1, x_1,0, x_1,1, x_2,0, x_2,1, s_0, s_1, s_2]
    where x_v,b are berth assignments (continuous [0,1]) and s_v are start times
    """
    n_vessels = len(vessels)
    n_berths = len(berths)
    
    # Extract variables
    berth_assignments = x[:n_vessels * n_berths].reshape(n_vessels, n_berths)
    start_times = x[n_vessels * n_berths:]
    
    total_cost = 0.0
    
    for v_idx, vessel in enumerate(vessels):
        # Calculate completion time
        completion_time = start_times[v_idx] + vessel.processing_time
        
        # Weighted completion time component
        weighted_completion = vessel.demurrage_cost * (completion_time - vessel.arrival_time)
        
        # Add penalty for fractional assignments (convexification)
        assignment_penalty = 0.0
        for b_idx in range(n_berths):
            assignment = berth_assignments[v_idx, b_idx]
            # Penalty for non-binary assignments
            assignment_penalty += 1000 * assignment * (1 - assignment)
        
        total_cost += weighted_completion + assignment_penalty
    
    return total_cost

def constraint_function(x):
    """Constraint function for convex optimization"""
    n_vessels = len(vessels)
    n_berths = len(berths)
    
    # Extract variables
    berth_assignments = x[:n_vessels * n_berths].reshape(n_vessels, n_berths)
    start_times = x[n_vessels * n_berths:]
    
    constraints = []
    
    # Constraint 1: Each vessel must be assigned to exactly one berth
    for v_idx in range(n_vessels):
        assignment_sum = np.sum(berth_assignments[v_idx, :])
        constraints.append(assignment_sum - 1.0)  # Should equal 0
    
    # Constraint 2: Non-overlap constraints for vessels at same berth
    for b_idx in range(n_berths):
        vessels_at_berth = []
        for v_idx in range(n_vessels):
            if berth_assignments[v_idx, b_idx] > 0.5:  # Consider vessels assigned to this berth
                vessels_at_berth.append(v_idx)
        
        # Add non-overlap constraints
        for i in range(len(vessels_at_berth)):
            for j in range(i + 1, len(vessels_at_berth)):
                v1_idx = vessels_at_berth[i]
                v2_idx = vessels_at_berth[j]
                
                # Either vessel 1 finishes before vessel 2 starts, or vice versa
                end_1 = start_times[v1_idx] + vessels[v1_idx].processing_time
                start_2 = start_times[v2_idx]
                
                end_2 = start_times[v2_idx] + vessels[v2_idx].processing_time
                start_1 = start_times[v1_idx]
                
                # Non-overlap constraint (using big-M relaxation)
                overlap_1 = max(0, end_1 - start_2)
                overlap_2 = max(0, end_2 - start_1)
                constraints.append(min(overlap_1, overlap_2))  # Should be 0
    
    return np.array(constraints)

print("Optimization functions defined!")

In [ ]:
# Set up optimization problem
n_vessels = len(vessels)
n_berths = len(berths)
n_variables = n_vessels * n_berths + n_vessels  # berth assignments + start times

# Initial guess (feasible starting point)
x0 = np.zeros(n_variables)

# Initialize berth assignments (round-robin)
for v_idx in range(n_vessels):
    berth_idx = v_idx % n_berths
    x0[v_idx * n_berths + berth_idx] = 1.0

# Initialize start times (at arrival times, adjusted for tidal windows)
for v_idx, vessel in enumerate(vessels):
    # Find earliest feasible start time
    earliest_feasible = vessel.arrival_time
    for t in np.arange(vessel.arrival_time, 24, 0.5):
        berth_idx = v_idx % n_berths
        if is_feasible_assignment(vessel, berths[berth_idx], t):
            earliest_feasible = t
            break
    x0[n_vessels * n_berths + v_idx] = earliest_feasible

# Variable bounds
# Berth assignments: [0, 1]
# Start times: [arrival_time, 24]
bounds = []
for v_idx in range(n_vessels):
    for b_idx in range(n_berths):
        bounds.append((0.0, 1.0))  # Berth assignment bounds

for v_idx, vessel in enumerate(vessels):
    bounds.append((vessel.arrival_time, 24.0))  # Start time bounds

print(f"Optimization setup: {n_variables} variables, {len(bounds)} bounds")
print(f"Initial objective value: {objective_function(x0):.2f}")

In [ ]:
# Solve the convex optimization problem
print("Solving convex optimization problem...")

# Use SLSQP method for constrained optimization
result = minimize(
    objective_function,
    x0,
    method='SLSQP',
    bounds=bounds,
    options={'maxiter': 1000, 'ftol': 1e-6}
)

print(f"Optimization status: {result.message}")
print(f"Success: {result.success}")
print(f"Objective value: {result.fun:.2f}")
print(f"Iterations: {result.nit}")

In [ ]:
# Extract and analyze the solution
def extract_solution(x):
    """Extract solution from optimization variables"""
    n_vessels = len(vessels)
    n_berths = len(berths)
    
    berth_assignments = x[:n_vessels * n_berths].reshape(n_vessels, n_berths)
    start_times = x[n_vessels * n_berths:]
    
    solution = []
    for v_idx, vessel in enumerate(vessels):
        # Find assigned berth (highest assignment value)
        assigned_berth = np.argmax(berth_assignments[v_idx, :])
        assignment_confidence = berth_assignments[v_idx, assigned_berth]
        
        start_time = start_times[v_idx]
        end_time = start_time + vessel.processing_time
        
        solution.append({
            'vessel_id': vessel.id,
            'assigned_berth': assigned_berth,
            'assignment_confidence': assignment_confidence,
            'start_time': start_time,
            'end_time': end_time,
            'processing_time': vessel.processing_time,
            'draft': vessel.draft,
            'demurrage_cost': vessel.demurrage_cost
        })
    
    return solution

if result.success:
    solution = extract_solution(result.x)
    
    print("\nOptimal Solution:")
    print("=" * 80)
    
    total_cost = 0.0
    for sol in solution:
        vessel = vessels[sol['vessel_id']]
        berth = berths[sol['assigned_berth']]
        
        # Calculate costs
        completion_time = sol['end_time']
        weighted_completion = vessel.demurrage_cost * (completion_time - vessel.arrival_time)
        total_cost += weighted_completion
        
        # Check feasibility
        is_feasible = is_feasible_assignment(vessel, berth, sol['start_time'])
        
        print(f"Vessel {sol['vessel_id']}:")
        print(f"  Assigned to Berth {sol['assigned_berth']} (confidence: {sol['assignment_confidence']:.3f})")
        print(f"  Start: {sol['start_time']:.1f}, End: {sol['end_time']:.1f}")
        print(f"  Draft: {sol['draft']}m, Required depth: {berth.depth_low_tide}m-{berth.depth_high_tide}m")
        print(f"  Feasible: {is_feasible}")
        print(f"  Cost: ${weighted_completion:,.0f}")
        print()
    
    print(f"Total Weighted Completion Time Cost: ${total_cost:,.0f}")
else:
    print("Optimization failed. Please check constraints and initial values.")

In [ ]:
# Create visualization of the solution
def visualize_solution(solution):
    """Create a Gantt chart visualization of the berth allocation"""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Colors for vessels
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
    
    # Plot 1: Gantt Chart
    ax1.set_title('Berth Allocation Schedule with Tidal Windows', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Time (hours)')
    ax1.set_ylabel('Berth Position')
    ax1.set_xlim(0, 24)
    ax1.set_ylim(-0.5, len(berths) - 0.5)
    ax1.set_yticks(range(len(berths)))
    ax1.set_yticklabels([f'Berth {i}' for i in range(len(berths))])
    ax1.grid(True, alpha=0.3)
    
    # Draw tidal windows
    for window in tidal_windows:
        ax1.axvspan(window.start_time, window.end_time, alpha=0.2, color='lightblue', 
                   label='High Tide Window' if window == tidal_windows[0] else '')
    
    # Draw vessel assignments
    for sol in solution:
        vessel_idx = sol['vessel_id']
        berth_idx = sol['assigned_berth']
        start_time = sol['start_time']
        end_time = sol['end_time']
        
        # Draw vessel as rectangle
        rect = patches.Rectangle((start_time, berth_idx - 0.3), 
                                end_time - start_time, 0.6,
                                linewidth=2, edgecolor='black',
                                facecolor=colors[vessel_idx % len(colors)],
                                alpha=0.7)
        ax1.add_patch(rect)
        
        # Add vessel label
        ax1.text((start_time + end_time) / 2, berth_idx, f'V{vessel_idx}',
                ha='center', va='center', fontweight='bold', fontsize=10)
    
    # Add arrival time markers
    for vessel in vessels:
        ax1.axvline(x=vessel.arrival_time, color='red', linestyle='--', alpha=0.5,
                   label='Arrival Time' if vessel == vessels[0] else '')
    
    ax1.legend(loc='upper right')
    
    # Plot 2: Depth Requirements vs Available Depth
    ax2.set_title('Draft Requirements vs Tidal Depth Availability', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Time (hours)')
    ax2.set_ylabel('Depth (meters)')
    ax2.set_xlim(0, 24)
    ax2.grid(True, alpha=0.3)
    
    # Plot tidal depth curves for each berth
    time_points = np.linspace(0, 24, 240)
    for berth_idx, berth in enumerate(berths):
        depths = [get_tidal_depth(t, berth) for t in time_points]
        ax2.plot(time_points, depths, linewidth=2, label=f'Berth {berth_idx} Available Depth')
    
    # Plot vessel draft requirements
    for sol in solution:
        vessel = vessels[sol['vessel_id']]
        start_time = sol['start_time']
        end_time = sol['end_time']
        
        # Draw draft requirement line during service period
        ax2.hlines(vessel.draft, start_time, end_time, 
                  colors=colors[sol['vessel_id'] % len(colors)],
                  linewidth=3, label=f'Vessel {sol["vessel_id"]} Draft')
        
        # Mark service period
        ax2.axvspan(start_time, end_time, alpha=0.1, 
                   color=colors[sol['vessel_id'] % len(colors)])
    
    ax2.legend(loc='lower right')
    
    plt.tight_layout()
    plt.show()

if result.success:
    visualize_solution(solution)

In [ ]:
# Performance analysis and sensitivity
def analyze_solution_quality(solution):
    """Analyze the quality of the obtained solution"""
    print("Solution Quality Analysis:")
    print("=" * 50)
    
    # Check feasibility of all assignments
    feasible_count = 0
    total_violations = 0
    
    for sol in solution:
        vessel = vessels[sol['vessel_id']]
        berth = berths[sol['assigned_berth']]
        
        if is_feasible_assignment(vessel, berth, sol['start_time']):
            feasible_count += 1
        else:
            total_violations += 1
    
    feasibility_rate = feasible_count / len(solution) * 100
    print(f"Feasibility Rate: {feasibility_rate:.1f}% ({feasible_count}/{len(solution)} vessels)")
    
    # Calculate berth utilization
    berth_utilization = {}
    for berth_idx in range(len(berths)):
        total_time = 0
        for sol in solution:
            if sol['assigned_berth'] == berth_idx:
                total_time += sol['processing_time']
        berth_utilization[berth_idx] = total_time / 24 * 100  # Percentage of 24 hours
    
    print("\nBerth Utilization:")
    for berth_idx, utilization in berth_utilization.items():
        print(f"  Berth {berth_idx}: {utilization:.1f}% ({berth_utilization[berth_idx]*24/100:.1f} hours)")
    
    # Calculate total cost breakdown
    total_cost = 0
    delay_costs = 0
    
    for sol in solution:
        vessel = vessels[sol['vessel_id']]
        completion_time = sol['end_time']
        delay = completion_time - vessel.arrival_time
        cost = vessel.demurrage_cost * delay
        
        total_cost += cost
        delay_costs += cost
    
    print(f"\nCost Breakdown:")
    print(f"  Total Delay Cost: ${delay_costs:,.0f}")
    print(f"  Average Cost per Vessel: ${delay_costs/len(solution):,.0f}")
    
    # Tidal window efficiency
    print(f"\nTidal Window Analysis:")
    for window_idx, window in enumerate(tidal_windows):
        window_duration = window.end_time - window.start_time
        utilized_time = 0
        
        for sol in solution:
            if window.start_time <= sol['start_time'] and sol['end_time'] <= window.end_time:
                utilized_time += sol['processing_time']
        
        efficiency = utilized_time / window_duration * 100
        print(f"  Window {window_idx + 1}: {efficiency:.1f}% utilized ({utilized_time:.1f}/{window_duration:.1f} hours)")

if result.success:
    analyze_solution_quality(solution)

### Why this Tier exists vs earlier Tiers
This Tier 1 provides the mathematical foundation for understanding the berth allocation with tidal windows problem. It establishes:

- **Theoretical optimality**: Mathematical formulation guarantees optimal solutions within the model assumptions
- **Problem insight**: Clear understanding of constraints, objectives, and feasible regions
- **Benchmark baseline**: Provides reference solutions for comparing heuristic and metaheuristic approaches
- **Convex relaxation**: Demonstrates how complex integer programs can be made computationally tractable

### Pros vs Cons vs other approaches

**Pros:**
- Guaranteed optimal solutions (when feasible)
- Mathematical rigor and provable bounds
- Clear problem formulation and constraint handling
- Sensitivity analysis capabilities

**Cons:**
- Computational complexity grows exponentially with problem size
- Requires problem simplifications (convex relaxation)
- Limited scalability for real-world large instances
- May struggle with highly non-convex constraints

### When to use this Tier
- Small to medium-sized problems (up to ~20 vessels)
- When optimality guarantees are required
- For benchmarking and validation of other methods
- Problems with well-structured, convexifiable constraints
- Academic and research settings where theoretical understanding is paramount